In [1]:
import itertools
import os

import joblib
import numpy as np
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier

SEED = 42
TARGET = 0.90
NS = (20, 30, 40)
FRACS = (0.25, 0.5, 1.0)
WEIGHT_ROOT = "./weights"

## Data

In [2]:
def load(name):
    """Load (X, y) arrays for a dataset."""
    return np.load(f"Datasets/{name}-X.npy"), np.load(f"Datasets/{name}-y.npy")


def split(X, y):
    """60% train, 20% validation, 20% test."""
    X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.4, random_state=SEED)
    X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=SEED)
    return X_train, y_train, X_val, y_val, X_test, y_test


def binarise(X):
    """Map each noisy +-1 feature to its sign."""
    return np.where(X > 0, 1.0, -1.0)

## Feature selection

In [3]:
def lookup_table_acc(S_train, y_train, S_val, y_val, cols):
    """Val accuracy of a majority-vote lookup table over the bits in `cols`."""
    w = 2 ** np.arange(len(cols))
    idx_tr = ((S_train[:, cols] > 0) * w).sum(1)
    idx_va = ((S_val[:, cols] > 0) * w).sum(1)
    count = np.bincount(idx_tr, minlength=2 ** len(cols))
    pos = np.bincount(idx_tr, weights=y_train, minlength=2 ** len(cols))
    table = (2 * pos > count).astype(int)  # unseen cells default to 0
    return (table[idx_va] == y_val).mean()


def select_features(S_train, y_train, S_val, y_val, min_gain=0.005):
    """Greedy forward selection that may add one bit or a pair of bits per step."""
    n = S_train.shape[1]
    score = lambda cols: lookup_table_acc(S_train, y_train, S_val, y_val, list(cols))
    selected, best = [], 0.5
    while True:
        rest = [j for j in range(n) if j not in selected]
        cands = [[j] for j in rest] + [list(p) for p in itertools.combinations(rest, 2)]
        acc, add = max((score(selected + c), c) for c in cands)
        if acc <= best + min_gain:
            break
        selected, best = selected + add, acc
        print(f"  + {add} -> {sorted(selected)}  val acc {acc:.4f}")
    return sorted(selected)

## Models

In [4]:
def tune(name, make, grid, train, val):
    """Fit `make(**params)` for every params in `grid`; keep the best on val (first wins ties)."""
    (Xtr, ytr), (Xva, yva) = train, val
    best = None
    for params in grid:
        model = make(**params).fit(Xtr, ytr)
        acc = model.score(Xva, yva)
        if best is None or acc > best[0]:
            best = (acc, params, model)
    print(f"  {name}: best {best[1]}  val acc {best[0]:.4f}")
    return best


forest_grid = [dict(n_estimators=n, max_depth=d, min_samples_leaf=l, max_features=m)
               for n in [100, 200] for d in [10, None]
               for l in [1, 3] for m in ["sqrt", None]]
xgb_grid = [dict(n_estimators=n, max_depth=d, learning_rate=r, min_child_weight=w)
            for n in [100, 200] for d in [8, 10]
            for r in [0.05, 0.1] for w in [1, 3]]

# (display name, weight folder, constructor, grid)
model_specs = [
    ("Random forest", "random_forest",
     lambda **p: RandomForestClassifier(random_state=SEED, n_jobs=-1, **p), forest_grid),
    ("Extra Trees", "extra_trees",
     lambda **p: ExtraTreesClassifier(random_state=SEED, n_jobs=-1, **p), forest_grid),
    ("XGBoost", "xgboost",
     lambda **p: XGBClassifier(random_state=SEED, n_jobs=-1, eval_metric="logloss",
                               verbosity=0, **p), xgb_grid),
]

## All datasets and scales

In [5]:
def run(n, frac):
    """Feature selection on the first `frac` of train/val, tune every tree model, save each best model."""
    X_train, y_train, X_val, y_val, X_test, y_test = split(*load(f"kryptonite-{n}"))
    m, mv = int(frac * len(y_train)), int(frac * len(y_val))
    S_tr, y_tr = binarise(X_train[:m]), y_train[:m]
    S_va, y_va = binarise(X_val[:mv]), y_val[:mv]
    S_te, y_te = binarise(X_test), y_test
    print(f"\n=== n={n}, {frac:.0%}: {m} train / {mv} val / {len(y_te)} test ===")

    cols = select_features(S_tr, y_tr, S_va, y_va)
    print(f"  selected {len(cols)} features: {cols}")
    F_tr, F_va, F_te = S_tr[:, cols], S_va[:, cols], S_te[:, cols]

    rows = []
    for model_name, folder, make, grid in model_specs:
        val_acc, params, model = tune(model_name, make, grid, (F_tr, y_tr), (F_va, y_va))
        test_acc = model.score(F_te, y_te)
        out_dir = os.path.join(WEIGHT_ROOT, folder, f"n{n}", f"{round(frac * 100)}pct")
        os.makedirs(out_dir, exist_ok=True)
        path = os.path.join(out_dir, "model.joblib")
        joblib.dump({"model": model, "features": cols, "params": params}, path)
        print(f"  {model_name:<14s} val {val_acc:.4f}  test {test_acc:.4f}  -> {path}")
        rows.append({"n": n, "frac": frac, "model": model_name, "train_size": m, "features": cols,
                     "params": params, "val": val_acc, "test": test_acc, "path": path})
    return rows


results = [row for n in NS for frac in FRACS for row in run(n, frac)]


=== n=20, 25%: 1800 train / 600 val / 2400 test ===
  + [4, 9] -> [4, 9]  val acc 0.5550
  + [10, 12] -> [4, 9, 10, 12]  val acc 0.6117
  + [5, 15] -> [4, 5, 9, 10, 12, 15]  val acc 0.6583
  + [1, 7] -> [1, 4, 5, 7, 9, 10, 12, 15]  val acc 0.7350
  + [0, 2] -> [0, 1, 2, 4, 5, 7, 9, 10, 12, 15]  val acc 0.8700
  selected 10 features: [0, 1, 2, 4, 5, 7, 9, 10, 12, 15]
  Random forest: best {'n_estimators': 100, 'max_depth': 10, 'min_samples_leaf': 1, 'max_features': None}  val acc 0.8850
  Random forest  val 0.8850  test 0.8892  -> ./weights\random_forest\n20\25pct\model.joblib
  Extra Trees: best {'n_estimators': 100, 'max_depth': 10, 'min_samples_leaf': 1, 'max_features': None}  val acc 0.8800
  Extra Trees    val 0.8800  test 0.8942  -> ./weights\extra_trees\n20\25pct\model.joblib
  XGBoost: best {'n_estimators': 200, 'max_depth': 8, 'learning_rate': 0.1, 'min_child_weight': 1}  val acc 0.8433
  XGBoost        val 0.8433  test 0.8554  -> ./weights\xgboost\n20\25pct\model.joblib

=== 

## Summary

In [6]:
print(f"{'n':>3} | {'subset':>6} | {'train':>6} | {'model':<14s} | {'#feat':>5} | {'val':>6} | {'test':>6}")
for r in results:
    flag = "" if r["test"] >= TARGET else "  < target"
    print(f"{r['n']:>3} | {r['frac']:>6.0%} | {r['train_size']:>6} | {r['model']:<14s} | "
          f"{len(r['features']):>5} | {r['val']:.4f} | {r['test']:.4f}{flag}")

  n | subset |  train | model          | #feat |    val |   test
 20 |    25% |   1800 | Random forest  |    10 | 0.8850 | 0.8892  < target
 20 |    25% |   1800 | Extra Trees    |    10 | 0.8800 | 0.8942  < target
 20 |    25% |   1800 | XGBoost        |    10 | 0.8433 | 0.8554  < target
 20 |    50% |   3600 | Random forest  |    10 | 0.9442 | 0.9475
 20 |    50% |   3600 | Extra Trees    |    10 | 0.9417 | 0.9408
 20 |    50% |   3600 | XGBoost        |    10 | 0.9175 | 0.9275
 20 |   100% |   7200 | Random forest  |    10 | 0.9558 | 0.9604
 20 |   100% |   7200 | Extra Trees    |    10 | 0.9558 | 0.9604
 20 |   100% |   7200 | XGBoost        |    10 | 0.9533 | 0.9592
 30 |    25% |   3000 | Random forest  |    10 | 0.9270 | 0.9273
 30 |    25% |   3000 | Extra Trees    |    10 | 0.9240 | 0.9137
 30 |    25% |   3000 | XGBoost        |    10 | 0.9000 | 0.9080
 30 |    50% |   6000 | Random forest  |    10 | 0.9550 | 0.9557
 30 |    50% |   6000 | Extra Trees    |    10 | 0.9540 | 0.

Reload every saved model and check it reproduces its test accuracy.

In [7]:
for r in results:
    saved = joblib.load(r["path"])
    _, _, _, _, X_test, y_test = split(*load(f"kryptonite-{r['n']}"))
    acc = saved["model"].score(binarise(X_test)[:, saved["features"]], y_test)
    assert abs(acc - r["test"]) < 1e-9, (r["path"], acc, r["test"])
    print(f"{r['path']} -> test {acc:.4f}")

./weights\random_forest\n20\25pct\model.joblib -> test 0.8892
./weights\extra_trees\n20\25pct\model.joblib -> test 0.8942
./weights\xgboost\n20\25pct\model.joblib -> test 0.8554
./weights\random_forest\n20\50pct\model.joblib -> test 0.9475
./weights\extra_trees\n20\50pct\model.joblib -> test 0.9408
./weights\xgboost\n20\50pct\model.joblib -> test 0.9275
./weights\random_forest\n20\100pct\model.joblib -> test 0.9604
./weights\extra_trees\n20\100pct\model.joblib -> test 0.9604
./weights\xgboost\n20\100pct\model.joblib -> test 0.9592
./weights\random_forest\n30\25pct\model.joblib -> test 0.9273
./weights\extra_trees\n30\25pct\model.joblib -> test 0.9137
./weights\xgboost\n30\25pct\model.joblib -> test 0.9080
./weights\random_forest\n30\50pct\model.joblib -> test 0.9557
./weights\extra_trees\n30\50pct\model.joblib -> test 0.9530
./weights\xgboost\n30\50pct\model.joblib -> test 0.9510
./weights\random_forest\n30\100pct\model.joblib -> test 0.9580
./weights\extra_trees\n30\100pct\model.jobli

## No feature selection (ablation)

The same models, grids, split and data fractions as above, but every model is tuned on all n sign features instead of the selected ones. The best model per (n, fraction) is saved to `weights/<model>_no_fs/n<n>/<pct>pct/model.joblib` in the same layout, with `features` set to all n columns.

In [8]:
def run_no_fs(n, frac):
    """Use the first `frac` of train/val with all n features, tune every model, save each best model."""
    X_train, y_train, X_val, y_val, X_test, y_test = split(*load(f"kryptonite-{n}"))
    m, mv = int(frac * len(y_train)), int(frac * len(y_val))
    S_tr, y_tr = binarise(X_train[:m]), y_train[:m]
    S_va, y_va = binarise(X_val[:mv]), y_val[:mv]
    S_te, y_te = binarise(X_test), y_test
    print(f"\n=== n={n}, {frac:.0%}, no FS: {m} train / {mv} val / {len(y_te)} test ===")

    cols = list(range(n))  # no feature selection: all features are used
    width = max(len(name) for name, *_ in model_specs)
    rows = []
    for model_name, folder, make, grid in model_specs:
        val_acc, params, model = tune(model_name, make, grid, (S_tr, y_tr), (S_va, y_va))
        test_acc = model.score(S_te, y_te)
        out_dir = os.path.join(WEIGHT_ROOT, f"{folder}_no_fs", f"n{n}", f"{round(frac * 100)}pct")
        os.makedirs(out_dir, exist_ok=True)
        path = os.path.join(out_dir, "model.joblib")
        joblib.dump({"model": model, "features": cols, "params": params}, path)
        print(f"  {model_name:<{width}s} val {val_acc:.4f}  test {test_acc:.4f}  -> {path}")
        rows.append({"n": n, "frac": frac, "model": model_name, "train_size": m, "features": cols,
                     "params": params, "val": val_acc, "test": test_acc, "path": path})
    return rows


results_no_fs = [row for n in NS for frac in FRACS for row in run_no_fs(n, frac)]


=== n=20, 25%, no FS: 1800 train / 600 val / 2400 test ===


  Random forest: best {'n_estimators': 200, 'max_depth': 10, 'min_samples_leaf': 1, 'max_features': None}  val acc 0.5850
  Random forest val 0.5850  test 0.5954  -> ./weights\random_forest_no_fs\n20\25pct\model.joblib
  Extra Trees: best {'n_estimators': 100, 'max_depth': 10, 'min_samples_leaf': 3, 'max_features': 'sqrt'}  val acc 0.5567
  Extra Trees   val 0.5567  test 0.5617  -> ./weights\extra_trees_no_fs\n20\25pct\model.joblib
  XGBoost: best {'n_estimators': 200, 'max_depth': 8, 'learning_rate': 0.1, 'min_child_weight': 3}  val acc 0.6000
  XGBoost       val 0.6000  test 0.5850  -> ./weights\xgboost_no_fs\n20\25pct\model.joblib

=== n=20, 50%, no FS: 3600 train / 1200 val / 2400 test ===
  Random forest: best {'n_estimators': 200, 'max_depth': 10, 'min_samples_leaf': 3, 'max_features': None}  val acc 0.6492
  Random forest val 0.6492  test 0.6296  -> ./weights\random_forest_no_fs\n20\50pct\model.joblib
  Extra Trees: best {'n_estimators': 100, 'max_depth': None, 'min_samples_leaf

### Summary without feature selection

In [9]:
width = max(len(name) for name, *_ in model_specs)
print(f"{'n':>3} | {'subset':>6} | {'train':>6} | {'model':<{width}s} | {'#feat':>5} | {'val':>6} | {'test':>6}")
for r in results_no_fs:
    flag = "" if r["test"] >= TARGET else "  < target"
    print(f"{r['n']:>3} | {r['frac']:>6.0%} | {r['train_size']:>6} | {r['model']:<{width}s} | "
          f"{len(r['features']):>5} | {r['val']:.4f} | {r['test']:.4f}{flag}")

  n | subset |  train | model         | #feat |    val |   test
 20 |    25% |   1800 | Random forest |    20 | 0.5850 | 0.5954  < target
 20 |    25% |   1800 | Extra Trees   |    20 | 0.5567 | 0.5617  < target
 20 |    25% |   1800 | XGBoost       |    20 | 0.6000 | 0.5850  < target
 20 |    50% |   3600 | Random forest |    20 | 0.6492 | 0.6296  < target
 20 |    50% |   3600 | Extra Trees   |    20 | 0.5925 | 0.5787  < target
 20 |    50% |   3600 | XGBoost       |    20 | 0.7292 | 0.7296  < target
 20 |   100% |   7200 | Random forest |    20 | 0.8367 | 0.8550  < target
 20 |   100% |   7200 | Extra Trees   |    20 | 0.7800 | 0.7896  < target
 20 |   100% |   7200 | XGBoost       |    20 | 0.8025 | 0.8167  < target
 30 |    25% |   3000 | Random forest |    30 | 0.6330 | 0.6055  < target
 30 |    25% |   3000 | Extra Trees   |    30 | 0.5990 | 0.5500  < target
 30 |    25% |   3000 | XGBoost       |    30 | 0.7330 | 0.7242  < target
 30 |    50% |   6000 | Random forest |    30 | 

Reload every saved no-FS model and check it reproduces its test accuracy.

In [10]:
for r in results_no_fs:
    saved = joblib.load(r["path"])
    _, _, _, _, X_test, y_test = split(*load(f"kryptonite-{r['n']}"))
    acc = saved["model"].score(binarise(X_test)[:, saved["features"]], y_test)
    assert abs(acc - r["test"]) < 1e-9, (r["path"], acc, r["test"])
    print(f"{r['path']} -> test {acc:.4f}")

./weights\random_forest_no_fs\n20\25pct\model.joblib -> test 0.5954
./weights\extra_trees_no_fs\n20\25pct\model.joblib -> test 0.5617
./weights\xgboost_no_fs\n20\25pct\model.joblib -> test 0.5850
./weights\random_forest_no_fs\n20\50pct\model.joblib -> test 0.6296
./weights\extra_trees_no_fs\n20\50pct\model.joblib -> test 0.5787
./weights\xgboost_no_fs\n20\50pct\model.joblib -> test 0.7296
./weights\random_forest_no_fs\n20\100pct\model.joblib -> test 0.8550
./weights\extra_trees_no_fs\n20\100pct\model.joblib -> test 0.7896
./weights\xgboost_no_fs\n20\100pct\model.joblib -> test 0.8167
./weights\random_forest_no_fs\n30\25pct\model.joblib -> test 0.6055
./weights\extra_trees_no_fs\n30\25pct\model.joblib -> test 0.5500
./weights\xgboost_no_fs\n30\25pct\model.joblib -> test 0.7242
./weights\random_forest_no_fs\n30\50pct\model.joblib -> test 0.7778
./weights\extra_trees_no_fs\n30\50pct\model.joblib -> test 0.5795
./weights\xgboost_no_fs\n30\50pct\model.joblib -> test 0.8097
./weights\random_